# Lab 3 — Anatomy of the Phase Autocorrelation Matrix

This lab merges and expands pedagogical ideas from Riccardo Gianpiccolo's original APM notebook with the phase-autocorrelation formulation used in Bruno Di Giorgi's rhythm-analysis software.

We use one notation consistently:

$$
P(\tau,\phi)=\sum_i x[\phi+i\tau]x[\phi+(i+1)\tau],\qquad 0\leq\phi<\tau.
$$

The first coordinate is candidate lag/period $\tau$; the second is phase $\phi$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from IPython.display import Audio, display

np.set_printoptions(precision=3, suppress=True)

def unbiased_autocorrelation(x, max_lag=None, normalize=True):
    x = np.asarray(x, dtype=float).reshape(-1)
    if max_lag is None:
        max_lag = len(x)-1
    max_lag = min(int(max_lag), len(x)-1)
    r = np.empty(max_lag+1)
    for tau in range(max_lag+1):
        a=x[:len(x)-tau]; b=x[tau:]
        r[tau] = np.dot(a,b)/max(1,len(a))
    if normalize and r[0] > 0:
        r = r/r[0]
    return r

def phase_autocorrelation(x, lags, unbiased=True):
    x=np.asarray(x,dtype=float).reshape(-1)
    lags=np.asarray(lags,dtype=int)
    P=np.zeros((len(lags), int(lags.max())))
    C=np.zeros_like(P)
    N=len(x)
    for ir,tau in enumerate(lags):
        for phi in range(tau):
            idx=np.arange(phi, N-tau, tau, dtype=int)
            if len(idx):
                P[ir,phi]=np.sum(x[idx]*x[idx+tau])
                C[ir,phi]=len(idx)
    if unbiased:
        return P/np.where(C==0,1,C), C
    return P, C

def frame_signal(x, frame_length, hop_length, center=True):
    x=np.asarray(x,float).reshape(-1)
    if center:
        p=frame_length//2
        x=np.pad(x,(p,p))
    starts=np.arange(0,max(1,len(x)-frame_length+1),hop_length,dtype=int)
    return np.stack([x[s:s+frame_length] for s in starts],axis=1), starts

def rhythmogram(x, frame_length=256, hop_length=32, lag_min=8, lag_max=96):
    frames, starts = frame_signal(x, frame_length, hop_length, center=True)
    lags=np.arange(lag_min,lag_max+1)
    R=np.zeros((len(lags),frames.shape[1]))
    w=np.hanning(frame_length)
    for j in range(frames.shape[1]):
        f=(frames[:,j]-frames[:,j].mean())*w
        ac=unbiased_autocorrelation(f, lag_max, normalize=False)
        R[:,j]=np.maximum(ac[lags],0)
        m=R[:,j].max()
        if m>0: R[:,j]/=m
    return R,lags,starts

def dp_tempo_path(R, lags, smoothness=30.0):
    R=np.asarray(R,float)
    eps=1e-10
    obs=-np.log(R+eps)
    nP,nT=R.shape
    cost=np.full((nP,nT),np.inf); back=np.zeros((nP,nT),dtype=int)
    cost[:,0]=obs[:,0]
    lp=np.log(np.asarray(lags,float))
    trans=smoothness*(lp[:,None]-lp[None,:])**2
    for t in range(1,nT):
        M=cost[:,t-1][None,:]+trans
        back[:,t]=np.argmin(M,axis=1)
        cost[:,t]=obs[:,t]+M[np.arange(nP),back[:,t]]
    path=np.zeros(nT,dtype=int); path[-1]=np.argmin(cost[:,-1])
    for t in range(nT-2,-1,-1): path[t]=back[path[t+1],t+1]
    return np.asarray(lags)[path]

def simple_novelty(x, fs, n_fft=1024, hop=256):
    f,t,Z=signal.stft(x,fs=fs,window='hann',nperseg=n_fft,noverlap=n_fft-hop,boundary=None,padded=False)
    mag=np.abs(Z)
    flux=np.maximum(mag[:,1:]-mag[:,:-1],0).sum(axis=0)
    flux=np.r_[0,flux]
    if flux.max()>0: flux/=flux.max()
    return flux,t

## 1. Ordinary autocorrelation as phase-collapsed APM

The APM keeps correlations belonging to different phase classes separate. Summing those classes recovers the ordinary lag correlation (up to the chosen bias/normalization convention).

In [ ]:
N=120
x=np.zeros(N); x[[5,25,45,65,85,105]]=1
lags=np.arange(5,41)
P,C=phase_autocorrelation(x,lags,unbiased=False)
row_sum=P.sum(axis=1)
r=unbiased_autocorrelation(x,40,normalize=False)
plt.figure(figsize=(10,3)); plt.plot(lags,row_sum,'o-',label='sum over phase of APM'); plt.plot(lags,r[lags]*(N-lags),'.--',label='ordinary product-sum'); plt.legend(); plt.xlabel('lag'); plt.show()
plt.figure(figsize=(9,6)); plt.imshow(P,origin='lower',aspect='auto',extent=[0,P.shape[1]-1,lags[0],lags[-1]]); plt.xlabel('phase φ'); plt.ylabel('lag τ'); plt.title('Phase autocorrelation matrix'); plt.colorbar(); plt.show()

## 2. Riccardo's influence-set idea

For one non-zero time instant $\hat t$, ask: **which APM cells can contain a product involving that instant?** The condition

$$
\hat t=\phi+i\tau
$$

defines families of lines in the $(\tau,\phi)$ plane. Two impulses generate two such families; visible APM energy occurs where compatible relationships between them exist.

In [ ]:
def influence_mask(t_hat,lags,N):
    M=np.zeros((len(lags),int(max(lags))))
    for ir,tau in enumerate(lags):
        for phi in range(tau):
            if t_hat>=phi and (t_hat-phi)%tau==0:
                i=(t_hat-phi)//tau
                if phi+i*tau<N: M[ir,phi]=1
    return M
lags2=np.arange(4,50)
I=influence_mask(73,lags2,120)
plt.figure(figsize=(9,6)); plt.imshow(I,origin='lower',aspect='auto',extent=[0,I.shape[1]-1,lags2[0],lags2[-1]]); plt.xlabel('phase φ'); plt.ylabel('lag τ'); plt.title('Influence set of one time instant'); plt.show()

## 3. Two impulses, then smoothing

Ideal impulses produce sparse intersections. Smoothing spreads each onset in time and turns those points into visible local structures, which is closer to what an onset-detection function produces.

In [ ]:
x2=np.zeros(160); x2[[20,60,100,140]]=1; x2[[72,124]]=1
lags3=np.arange(10,70)
P0,_=phase_autocorrelation(x2,lags3,unbiased=True)
xs=gaussian_filter1d(x2,1.8)
Ps,_=phase_autocorrelation(xs,lags3,unbiased=True)
for M,title in [(P0,'ideal impulses'),(Ps,'smoothed events')]:
    plt.figure(figsize=(8,5)); plt.imshow(M,origin='lower',aspect='auto',extent=[0,M.shape[1]-1,lags3[0],lags3[-1]]); plt.xlabel('phase φ'); plt.ylabel('lag τ'); plt.title(title); plt.colorbar(); plt.show()

## 4. A cyclostationary / polyphase viewpoint

There is a useful conceptual analogy with cyclostationarity. For a candidate period $\tau$, split the sequence into phase classes

$$
x[\phi],\;x[\phi+\tau],\;x[\phi+2\tau],\ldots
$$

for $\phi=0,\ldots,\tau-1$. The APM asks how strongly adjacent samples **within each phase class** support repetition at that candidate period. Ordinary autocorrelation collapses those phase classes together.

This is an analogy, not an assertion that the APM is identical to the full theory of cyclic autocorrelation/spectral correlation.

In [ ]:
tau=8
pat=np.array([1,0,0,1,0,0,1,0],float)
xp=np.tile(pat,6)
classes=[xp[phi::tau] for phi in range(tau)]
fig,axes=plt.subplots(tau,1,figsize=(10,8),sharex=True)
for phi,(ax,c) in enumerate(zip(axes,classes)):
    ax.stem(np.arange(len(c)),c,basefmt=' '); ax.set_ylabel(str(phi),rotation=0,labelpad=10)
axes[-1].set_xlabel('cycle index'); fig.suptitle('Polyphase classes for candidate period τ=8'); plt.tight_layout(); plt.show()
P8,_=phase_autocorrelation(xp,np.array([8]),unbiased=False)
print('P(8, φ)=',P8[0,:8])

## 5. Translation: same period, different phase

Shift the rhythm. The ordinary autocorrelation preserves essentially the same lag structure, while APM energy moves in the phase coordinate. This is the cleanest demonstration of why phase resolution matters.

In [ ]:
shift=2
xq=np.roll(xp,shift)
rp=unbiased_autocorrelation(xp,24); rq=unbiased_autocorrelation(xq,24)
Pp,_=phase_autocorrelation(xp,np.array([8]),unbiased=False)
Pq,_=phase_autocorrelation(xq,np.array([8]),unbiased=False)
fig,ax=plt.subplots(1,2,figsize=(11,3))
ax[0].plot(rp,label='original'); ax[0].plot(rq,'--',label='shifted'); ax[0].set_title('Autocorrelation'); ax[0].legend()
ax[1].stem(np.arange(8),Pp[0,:8],linefmt='C0-',markerfmt='C0o',basefmt=' '); ax[1].stem(np.arange(8),Pq[0,:8],linefmt='C1-',markerfmt='C1s',basefmt=' '); ax[1].set_title('APM slice at τ=8'); ax[1].set_xlabel('phase φ'); plt.tight_layout(); plt.show()

## 6. Experiments

- change the translation amount;
- delete one onset;
- add timing jitter before quantization;
- mix two periods;
- compare biased and unbiased normalization.

**Interpretive question:** which changes affect period, which affect phase, and which affect both?

### Attribution

The influence-set decomposition and its geometric teaching interpretation are based on material developed by **Riccardo Gianpiccolo**. The phase-autocorrelation representation and short-time rhythm-analysis line are connected to software and research developed by **Bruno Di Giorgi** during his PhD work at Politecnico di Milano. This notebook is a new teaching synthesis for CMRM.